# Distillation of GPT-2 Small

## 1. Imports and Configuration

In [ ]:
import os
import math
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast
import matplotlib.pyplot as plt


def set_seed(seed):
    """Make results reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


CFG = {
    # Data
    "dataset_name": "Salesforce/wikitext",
    "dataset_config": "wikitext-2-raw-v1",  # ~2.4M tokens; the only dataset used in this project
    "block_size": 1024,      # tokens per text block
    "batch_size": 16,        # blocks per training step (~32 GB peak VRAM at block_size 1024)

    # Teacher and student model size
    "teacher_name": "gpt2",
    "student_n_layer": 6,
    "student_n_embd": 384,
    "student_n_head": 6,

    # Training
    "seed": 42,
    # 16 x 1024 = 16,384 tokens per optimizer step, close to the 32 x 384 = 12,288 that this
    # learning rate was originally tuned for, so 5e-4 still applies.
    "lr": 5e-4,
    "epochs": 15,
    "sweep_epochs": 5,
    "early_stopping_patience": 2,  # stop if validation perplexity does not improve for this many epochs

    # Distillation
    "temperature": 2.0,
    "temperature_values": [1.0, 2.0, 4.0, 7.0, 10.0, 15.0],
    "alpha": 0.5,
}

set_seed(CFG["seed"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load the Tokenizer

In [ ]:
# Load the GPT-2 tokenizer
# tokenized returns token IDs, attention masks, etc. as a dictionary of lists.
tokenizer = GPT2TokenizerFast.from_pretrained(CFG["teacher_name"])

print("Vocabulary size:", tokenizer.vocab_size)
# Quick sanity check: encode then decode a sentence and confirm we get it back.
demo = tokenizer("This project is a demonstration of knowledge distillation for GPT-2")
print("Token IDs:", demo["input_ids"])
print("Decoded   :", tokenizer.decode(demo["input_ids"]))

## 3. Load & Prepare the Data

We use **`wikitext-2-raw-v1`** for everything in this project: the temperature sweep, the final
training of all three students, and all evaluation. It is about 2.4M GPT-2 tokens, which is small
enough to train 11 models within a reasonable time budget.

The preparation has three steps:

1. **Tokenize** every line of text into token IDs.
2. **Group into fixed-length blocks** of block_size tokens. Language models train on
   fixed-length sequences, so we concatenate all the tokens into one long stream and then chop
   it into equal chunks. Any leftover tokens that don't fill a full block are dropped.
3. **Create DataLoader** for training, validation and testing

In [ ]:
# 1) Download the raw dataset. It has "train", "validation", and "test" sets.
raw_datasets = load_dataset(CFG["dataset_name"], CFG["dataset_config"])
print(raw_datasets)

# 2) Tokenize text in batches.
# getting this warning: Token indices sequence length is longer than the specified maximum sequence length for this model (1063 > 1024)
# it is ok, the tokens are regrouped into blocks of CFG["block_size"] below
def tokenize_batch(batch):
    return tokenizer(batch["text"])

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)
print(tokenized_datasets)

# 3) Join tokenized text together and split it into fixed-size blocks.
block_size = CFG["block_size"]

def group_texts(batch):
    all_input_ids = []

    for token_list in batch["input_ids"]:
        all_input_ids.extend(token_list)

    total_length = (len(all_input_ids) // block_size) * block_size
    blocks = []

    for i in range(0, total_length, block_size):
        blocks.append(all_input_ids[i:i + block_size])

    return {"input_ids": blocks}

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_datasets["train"].column_names,
)
lm_datasets.set_format(type="torch", columns=["input_ids"])

print(lm_datasets)

# 4) Create DataLoaders for each split. Each batch is a tensor with shape (batch_size, block_size).
train_loader = DataLoader(lm_datasets["train"], batch_size=CFG["batch_size"], shuffle=True)
val_loader = DataLoader(lm_datasets["validation"], batch_size=CFG["batch_size"])
test_loader = DataLoader(lm_datasets["test"], batch_size=CFG["batch_size"])

## 4. Load the Teacher (GPT-2 Small)

The teacher is a pretrained GPT-2 small

In [ ]:
def count_parameters(model):
    total = 0
    for param in model.parameters():
        total += param.numel()
    return total

# Download/load GPT-2 small.
teacher = GPT2LMHeadModel.from_pretrained(CFG["teacher_name"]).to(device)
teacher.eval()  # freeze behaviour: no dropout, no parameter updates

teacher_params = count_parameters(teacher)
print(f"Teacher: {CFG['teacher_name']}")
print(f"Teacher parameters: {teacher_params}")
print(f"Teacher layers: {teacher.config.n_layer}, hidden size: {teacher.config.n_embd}")

## 5. Build Three Student Models (Same Architecture)

We create three students with the **same smaller GPT-2 architecture** so the comparison is fair.
Only the training objective changes:

- **Teacher + corpus:** learns from both hard labels and teacher soft labels.
- **Corpus only:** learns only from the true next tokens in WikiText.
- **Teacher only:** learns only by matching the teacher's output distribution.

All three students use the same tokenizer and vocabulary as the teacher, which is required for
comparing next-token distributions.

In [ ]:
def build_student():
    student_config = GPT2Config(
        vocab_size=teacher.config.vocab_size,    # MUST match the teacher,number if tokens in the vocabulary
        n_positions=teacher.config.n_positions,  # max context length (1024)
        n_ctx=teacher.config.n_positions,
        n_embd=CFG["student_n_embd"],            # hidden size (width)
        n_layer=CFG["student_n_layer"],          # number of transformer blocks (depth)
        n_head=CFG["student_n_head"],            # attention heads
        bos_token_id=tokenizer.bos_token_id,     # token ID for the beginning-of-sequence token
        eos_token_id=tokenizer.eos_token_id,     # token ID for the end-of-sequence token
    )
    return GPT2LMHeadModel(student_config).to(device)


def build_seeded_student():
    # Build each student from the same initial weights for a fair comparison.
    set_seed(CFG["seed"])
    return build_student()


student_models = {
    "Teacher + corpus": build_seeded_student(),
    "Corpus only": build_seeded_student(),
    "Teacher only": build_seeded_student(),
}

student_params = count_parameters(student_models["Teacher + corpus"]) # all students have the same number of parameters, so we can just count one of them
print(f"Student parameters: {student_params:,}")
print(f"Student layers: {CFG['student_n_layer']}, hidden size: {CFG['student_n_embd']}")
print(f"Compression ratio (teacher / student): {teacher_params / student_params:.1f}x")

## 6. Loss Functions For The Three Training Methods

This project compares three ways to train the same student architecture:

- **Teacher + corpus:** uses both the real corpus labels and the teacher's soft targets.
- **Corpus only:** uses only the real next-token labels from the dataset.
- **Teacher only:** uses only the teacher's soft targets.

In code, the combined method is:

```text
total_loss = alpha * corpus_loss + (1 - alpha) * teacher_loss
```

The loss design follows ideas from Hinton, Vinyals, and Dean's paper **Distilling the Knowledge in a Neural Network**:

**Hard and soft losses**

- `corpus_loss` is the hard-label loss. It uses cross-entropy against the true next token from the dataset.
- `teacher_loss` is the soft-label loss. It uses KL divergence to match the teacher's probability distribution.
- The paper explains that combining hard labels and soft teacher targets can improve the student.

**Temperature**

- Temperature softens the teacher's probability distribution.
- A higher temperature makes the probabilities less sharp and reveals more information about the teacher's uncertainty.
- In this project, we run a temperature sweep before final training to choose a good value.

**T squared scaling**

- The paper notes that gradients from soft targets shrink when temperature increases.
- To compensate, the distillation loss is multiplied by `temperature ** 2`.
- In the code, this scaling is applied inside `teacher_loss`.


In [ ]:
def compute_training_loss(student_logits, teacher_logits, input_ids, method):
    # This function computes the loss for one training batch.

    # student_logits: (batch_size, block_size, vocab_size) tensor of scores for every possible next token
    # we extract the vocab_size
    vocab_size = student_logits.size(-1)

    # GPT predicts the next token, so we compare position 0 with label 1, position 1 with label 2, etc.
    # take all batches, take all positions except the last one, take all vocabulary scores
    student_next_token_scores = student_logits[:, :-1, :].contiguous()
    # take all batches, take tokens from position 1 onward
    true_next_tokens = input_ids[:, 1:].contiguous()

    # Corpus loss: student vs. the real next token from the dataset.
    # corpus_loss = average of -log(student probability assigned to each true next token)
    # cross entropy expect raw logits
    corpus_loss = F.cross_entropy(
        student_next_token_scores.view(-1, vocab_size), # reshape [batch_size * (block_size - 1), vocab_size]
        true_next_tokens.view(-1),                      # reshape [batch_size * (block_size - 1)]
    )

    # Teacher loss: student vs. the teacher's full probability distribution.
    teacher_loss = torch.zeros((), device=student_logits.device)
    if teacher_logits is not None: ## no teacher_logits for corpus only training
        teacher_next_token_scores = teacher_logits[:, :-1, :].contiguous()

        student_log_probs = F.log_softmax(student_next_token_scores / CFG["temperature"], dim=-1)
        teacher_probs = F.softmax(teacher_next_token_scores / CFG["temperature"], dim=-1)

        teacher_loss = F.kl_div(
            student_log_probs.view(-1, vocab_size),
            teacher_probs.view(-1, vocab_size),
            reduction="batchmean",
        ) * (CFG["temperature"] ** 2)

    # Choose which training method this student uses.
    if method == "combined":
        total_loss = CFG["alpha"] * corpus_loss + (1 - CFG["alpha"]) * teacher_loss
    elif method == "corpus_only":
        total_loss = corpus_loss
    elif method == "teacher_only":
        total_loss = teacher_loss
    else:
        raise ValueError(f"Unknown training method: {method}")

    return total_loss, corpus_loss, teacher_loss

## 7. Training

### 7.1 Perplexity Evaluation

Perplexity measures how well the model predicts the next token. It is calculated from the average cross-entropy loss:

```text
perplexity = exp(average cross-entropy loss)
```

In [ ]:
def evaluate_perplexity(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)

            # Hugging Face computes the cross-entropy internally when labels are provided.
            # outputs.loss is the average negative log probability of the correct next tokens.
            outputs = model(input_ids=input_ids, labels=input_ids)

            num_tokens = input_ids.size(0) * (input_ids.size(1) - 1)
            total_loss = total_loss + outputs.loss.item() * num_tokens
            total_tokens = total_tokens + num_tokens

    average_loss = total_loss / total_tokens  # average log perplexity
    return math.exp(average_loss)  # convert back to perplexity

### 7.2 Model Training

Main model training loop with early stop

In [ ]:
def train_model(model, name, method):
    # This function trains one student model and records average loss/perplexity per epoch.
    # It stops early if validation perplexity stops improving.

    # Reseed here so every method sees the same shuffled batch order and the same dropout masks.
    # Without this, the three students would differ by data order as well as by loss function,
    # and that difference can be as large as the effect we are trying to measure.
    set_seed(CFG["seed"])

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
    loss_history = []
    validation_history = []

    best_validation_ppl = None
    epochs_without_improvement = 0

    for epoch in range(CFG["epochs"]):
        model.train()
        progress = tqdm(train_loader, desc=f"{name}: epoch {epoch + 1}/{CFG['epochs']}")

        total_epoch_loss = 0
        num_batches = 0

        for batch in progress:
            input_ids = batch["input_ids"].to(device)

            # Run the student, get the final output scores (logits) for every possible next token.
            student_logits = model(input_ids=input_ids).logits

            # Run the teacher only when this method needs teacher guidance.
            teacher_logits = None
            if method in ["combined", "teacher_only"]:
                with torch.no_grad():
                    teacher_logits = teacher(input_ids=input_ids).logits

            loss, corpus_loss, teacher_loss = compute_training_loss(
                student_logits,
                teacher_logits,
                input_ids,
                method,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_epoch_loss = total_epoch_loss + loss.item()
            num_batches = num_batches + 1
            progress.set_postfix(loss=f"{loss.item():.3f}")

        average_epoch_loss = total_epoch_loss / num_batches
        loss_history.append(average_epoch_loss)
        print(f"{name} average training loss after epoch {epoch + 1}: {average_epoch_loss:.3f}")

        validation_ppl = evaluate_perplexity(model, val_loader)
        validation_history.append(validation_ppl)
        print(f"{name} validation perplexity after epoch {epoch + 1}: {validation_ppl:.2f}")

        if best_validation_ppl is None or validation_ppl < best_validation_ppl:
            best_validation_ppl = validation_ppl
            epochs_without_improvement = 0
        else:
            epochs_without_improvement = epochs_without_improvement + 1

        if epochs_without_improvement >= CFG["early_stopping_patience"]:
            print(f"Early stopping {name}: validation perplexity did not improve.")
            break

    print(f"Finished training {name} model.")
    return loss_history, validation_history

### 7.3 Temperature Sweep Before Final Training
- Before final training, we try several temperature values: 1, 2, 4, 7, 10, 15, 20, 30.
- Each sweep run trains a throwaway student with `method="teacher_only"`, so the teacher's soft targets are the only training signal.
- The sweep uses validation perplexity to choose a good temperature.
- After the sweep, `CFG["temperature"]` is set to the best temperature.
- Fewer epochs are used to save run time, so we expect these perplexities to be higher than the final models.

**Why the sweep uses `teacher_only` instead of `Teacher + corpus`**

Temperature only affects `teacher_loss`. If the sweep trained with the combined objective, then with
`alpha = 0.5` half of every gradient would come from `corpus_loss` regardless of the temperature.
That corpus signal is identical for every temperature, so it dilutes the thing we are trying to
measure and can flatten the sweep curve into noise. Training on the teacher signal alone makes the
sweep a direct measurement of which temperature transfers the most usable knowledge from the teacher.

We still rank temperatures by validation perplexity, which is measured against the real next tokens.
So the question the sweep answers is: *at which temperature does imitating the teacher best transfer
to real next-token prediction?*

The trade-off is that the temperature is selected under one objective and then used by the final
`Teacher + corpus` student under another, so the winner is not guaranteed to be optimal for the
blended loss. We accept that in exchange for a temperature signal that is actually measurable at this
scale. The absolute perplexities in the table below will also be worse than the final students',
since these runs never see the corpus labels.

In [ ]:
temperature_values = CFG["temperature_values"]
original_epochs = CFG["epochs"]
sweep_epochs = CFG["sweep_epochs"]

sweep_rows = []

for temperature in temperature_values:
    print(f"\n=== Temperature sweep: T = {temperature} ===")

    CFG["temperature"] = temperature
    CFG["epochs"] = sweep_epochs

    sweep_student = build_seeded_student()
    # Sweep with the teacher signal alone so the corpus loss does not dilute the temperature effect.
    sweep_loss_history, sweep_validation_history = train_model(
        sweep_student,
        name=f"Temperature {temperature}",
        method="teacher_only",
    )

    best_validation_ppl = min(sweep_validation_history)
    final_validation_ppl = sweep_validation_history[-1]

    sweep_rows.append(
        {
            "Temperature": temperature,
            # This loss is KL * T^2, so it grows with T by construction and must not be compared
            # across rows. Use the validation perplexity columns to rank temperatures.
            "Final training loss (KL x T^2)": sweep_loss_history[-1],
            "Best validation perplexity": best_validation_ppl,
            "Final validation perplexity": final_validation_ppl,
        }
    )

    del sweep_student
    if device == "cuda":
        torch.cuda.empty_cache()

# Pick the temperature with the lowest best validation perplexity.
temperature_sweep = pd.DataFrame(sweep_rows)
best_temperature = temperature_sweep.loc[temperature_sweep["Best validation perplexity"].idxmin(), "Temperature"]

# Restore the normal epoch count, but keep the best temperature for final training.
CFG["epochs"] = original_epochs
CFG["temperature"] = float(best_temperature)

print(f"\nBest temperature from sweep: {CFG['temperature']}")
temperature_sweep

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(
    temperature_sweep["Temperature"],
    temperature_sweep["Best validation perplexity"],
    marker="o",
)
plt.title("Temperature sweep: best validation perplexity")
plt.xlabel("Temperature")
plt.ylabel("Best validation perplexity (lower is better)")
plt.grid(True)
plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/temperature_sweep.png", dpi=150)
plt.show()

### 7.4 Train All 3 Student Models
- Teacher + corpus
- Corpus only
- Teacher only

In [ ]:
# Train all three final student variants using the best temperature from the sweep.
print(f"Final student training will use temperature = {CFG['temperature']}")

training_methods = {
    "Teacher + corpus": "combined",
    "Corpus only": "corpus_only",
    "Teacher only": "teacher_only",
}

student_loss_history = {}
student_validation_history = {}

for name, method in training_methods.items():
    print(f"\n=== Training: {name} ===")
    loss_history, validation_history = train_model(student_models[name], name, method)
    student_loss_history[name] = loss_history
    student_validation_history[name] = validation_history

## 8. Save The Trained Student Models

We save each trained student separately so the three methods can be reloaded or inspected later.

In [ ]:
checkpoint_dirs = {
    "Teacher + corpus": "checkpoints/student_teacher_plus_corpus",
    "Corpus only": "checkpoints/student_corpus_only",
    "Teacher only": "checkpoints/student_teacher_only",
}

for name, path in checkpoint_dirs.items():
    os.makedirs(path, exist_ok=True)
    student_models[name].save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f"Saved {name} -> {path}")

## 9. Evaluation & Analysis

We now compare the **teacher** and all **three student training methods**.

### 9.1 Validation and test perplexity
Lower is better. Validation perplexity helps check overfitting, while test perplexity is the final performance number.

In [ ]:
# Measure validation and test perplexity.
teacher_val_ppl = evaluate_perplexity(teacher, val_loader)
teacher_ppl = evaluate_perplexity(teacher, test_loader)

student_val_ppls = {}
student_ppls = {}

for name, model in student_models.items():
    student_val_ppls[name] = evaluate_perplexity(model, val_loader)
    student_ppls[name] = evaluate_perplexity(model, test_loader)

print(f"Teacher validation perplexity: {teacher_val_ppl:.2f}")
print(f"Teacher test perplexity: {teacher_ppl:.2f}")

for name in student_models:
    print(f"{name} validation perplexity: {student_val_ppls[name]:.2f}")
    print(f"{name} test perplexity: {student_ppls[name]:.2f}")

### 9.2 Generated text quality
We feed the **same prompts** with deterministic decoding to the teacher and each student, then compare fluency, coherence, and repetition.

In [ ]:
def generate_text(model, prompt, max_new_tokens=40):
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic generation for fair comparison
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

all_models = {"Teacher (GPT-2)": teacher, **student_models}

prompts = [
    "The history of artificial intelligence",
    "The city was famous for its",
    "Climate change is one of the most important",
    "A good machine learning model should",
    "The scientist opened the notebook and discovered",
    "In simple terms, gravity is",
    "The main advantage of renewable energy is",
    "During the experiment, the researchers observed",
    "Once upon a time in a small village",
]

for prompt in prompts:
    print("=" * 90)
    print(f"PROMPT: {prompt}\n")
    for name, model in all_models.items():
        print(f"[{name}] {generate_text(model, prompt)}\n")

### 9.3 Inference speed
We time how long each model takes to generate a fixed number of tokens and report **tokens per
second**. Higher is better.

In [ ]:
def measure_speed(model, n_tokens=100, n_runs=5):
    model.eval()
    input_ids = tokenizer("The", return_tensors="pt").input_ids.to(device)

    def one_run():
        with torch.no_grad():
            model.generate(
                input_ids,
                max_new_tokens=n_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

    # The first generation call is often slower than normal because PyTorch/GPU may need to do one-time setup
    one_run()
    if device == "cuda":
        torch.cuda.synchronize() # forces Python to wait until the GPU is done

    start = time.time()
    for _ in range(n_runs):
        one_run()
    if device == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - start
    return (n_tokens * n_runs) / elapsed


teacher_tps = measure_speed(teacher)
student_tps = {
    name: measure_speed(model)
    for name, model in student_models.items()
}

print(f"Teacher: {teacher_tps:.1f} tokens/sec")
for name, tps in student_tps.items():
    print(f"{name}: {tps:.1f} tokens/sec ({tps / teacher_tps:.1f}x teacher speed)")

### 9.4 Model size & parameter count
Finally we report parameter counts, on-disk size, perplexity, and speed in one summary table.

In [ ]:
def dir_size_mb(path) -> float:
    """Total size (in MB) of the saved-model files in a folder."""
    total_bytes = 0
    for root, _dirs, files in os.walk(path):
        for f in files:
            total_bytes += os.path.getsize(os.path.join(root, f))
    return total_bytes / (1024 ** 2)


teacher_size_mb = teacher_params * 4 / (1024 ** 2)  # teacher model is not saved, so estimate: float32 weights

rows = [
    {
        "Model": "Teacher (GPT-2)",
        "Training method": "pretrained teacher",
        "Parameters (M)": teacher_params / 1e6,
        "Disk size (MB)": teacher_size_mb,
        "Validation perplexity": teacher_val_ppl,
        "Test perplexity": teacher_ppl,
        "Tokens/sec": teacher_tps,
    }
]

for name in student_models:
    rows.append(
        {
            "Model": f"Student ({name})",
            "Training method": training_methods[name],
            "Parameters (M)": student_params / 1e6,
            "Disk size (MB)": dir_size_mb(checkpoint_dirs[name]),
            "Validation perplexity": student_val_ppls[name],
            "Test perplexity": student_ppls[name],
            "Tokens/sec": student_tps[name],
        }
    )

summary = pd.DataFrame(rows).round(2)
display(summary)

## 10. Plots & Conclusions

Visualising the numbers makes the training-method comparison clearer. We plot:

1. Training loss per epoch for all three students.
2. Validation perplexity per epoch for all three students.
3. Final test perplexity for teacher and students.
4. Inference speed for teacher and students.

Plots 1 and 2 both show training progress, but only plot 2 can be compared across methods. The three
training losses are different quantities: `Corpus only` reports a cross-entropy, `Teacher only`
reports `KL * T^2`, and `Teacher + corpus` reports a weighted blend of the two. A lower curve in
plot 1 therefore does **not** mean a better model. Validation perplexity in plot 2 is measured the
same way for every student, so it is the curve to read for quality, and it is also the metric early
stopping selects on. The circled point on each line marks that student's best epoch.

In [ ]:
os.makedirs("results", exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

max_epochs_plotted = max(len(losses) for losses in student_loss_history.values())

# (1) Training loss per epoch. The three curves use different loss definitions, so they show each
# student's own progress but must not be compared with each other.
for name, losses in student_loss_history.items():
    epochs_axis = range(1, len(losses) + 1)
    axes[0].plot(epochs_axis, losses, marker="o", label=name)
axes[0].set_title("Training loss per epoch (scales differ per method)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Average training loss")
axes[0].set_xticks(range(1, max_epochs_plotted + 1))
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# (2) Validation perplexity per epoch, on a raw (linear) scale. Measured identically for every
# method, so this is the curve to compare across students.
for name, ppls in student_validation_history.items():
    epochs_axis = range(1, len(ppls) + 1)
    line, = axes[1].plot(epochs_axis, ppls, marker="o", label=name)
    best_epoch = int(np.argmin(ppls)) + 1
    axes[1].scatter(
        [best_epoch], [min(ppls)],
        s=160, facecolors="none", edgecolors=line.get_color(), linewidths=2, zorder=5,
    )
axes[1].axhline(teacher_val_ppl, color="gray", linestyle="--", linewidth=1.5, label="Teacher (GPT-2)")
axes[1].set_title("Validation perplexity per epoch (lower is better)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation perplexity")
axes[1].set_xticks(range(1, max_epochs_plotted + 1))
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# (3) Final test perplexity.
plot_names = ["Teacher"] + list(student_models.keys())
plot_ppl = [teacher_ppl] + [student_ppls[name] for name in student_models]
axes[2].bar(plot_names, plot_ppl)
axes[2].set_title("Test perplexity (lower is better)")
axes[2].set_ylabel("Perplexity")
axes[2].tick_params(axis="x", rotation=25)

# (4) Inference speed.
plot_tps = [teacher_tps] + [student_tps[name] for name in student_models]
axes[3].bar(plot_names, plot_tps)
axes[3].set_title("Inference speed (higher is better)")
axes[3].set_ylabel("Tokens / sec")
axes[3].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150)
plt.show()